# 11. Umbrales de clasificación de apnea

## Objetivo y reglas

Definir los puntos de corte utilizando exclusivamente desarrollo.

Se mantienen las decisiones anteriores:
- Marcas QRS de la base.
- Sin Wiener.
- Minutos con etiqueta válida, características calculables y no más
  del 20 % de RR corregidos.
- F: log de la potencia cruzada RR–EDR en 0,01–0,05 Hz,
  EDR por área, ventana de 240 s y Welch de tres segmentos.
- HRV de referencia: potencia VLF de Lomb, ventana de 180 s.

La referencia es el umbral que maximiza el índice de Youden.
Se evaluarán las alternativas previstas: mediana de los Youden
obtenidos por bootstrap de sujetos, umbral que iguala la proporción
total de apnea estimada con la real y combinaciones AND/OR.

Las comparaciones entre F y HRV utilizarán los mismos minutos.
Las alternativas se evaluarán dejando sujetos afuera; no se adoptarán
por mejorar solamente sobre los datos usados para ajustar el umbral.

El conjunto de prueba permanece reservado.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os
import sys

ROOT = Path("/content/drive/MyDrive/TP-PSB-G8-main")
assert (ROOT / "src").is_dir(), f"No se encontró el proyecto: {ROOT}"

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%pip install -q -r requirements.txt

import numpy as np
import pandas as pd

from src.config import SPLIT_CSV
from src.stats.thresholds import roc_youden, record_agreement

split = pd.read_csv(SPLIT_CSV)
dev = split.loc[split["set"].eq("dev"), ["record", "subject"]]

RUTA_E3 = ROOT / "results" / "e3"

for nombre in ("acoplamiento.parquet", "hrv.parquet"):
    assert (RUTA_E3 / nombre).exists(), f"Falta {nombre}"

print(f"Desarrollo: {len(dev)} registros, {dev['subject'].nunique()} sujetos")

Mounted at /content/drive
Desarrollo: 47 registros, 19 sujetos


In [2]:
# Leemos solo las columnas necesarias de las tablas ya calculadas.
coup = pd.read_parquet(
    RUTA_E3 / "acoplamiento.parquet",
    columns=[
        "record", "minute", "label", "corrected",
        "edr", "window", "estimator", "xp_ap_l",
    ],
)

hrv = pd.read_parquet(
    RUTA_E3 / "hrv.parquet",
    columns=["record", "minute", "label", "corrected", "window", "vlf"],
)

f_tab = coup.loc[
    coup["record"].isin(dev["record"])
    & coup["edr"].eq("area")
    & coup["window"].eq(240)
    & coup["estimator"].eq("welch3"),
    ["record", "minute", "label", "corrected", "xp_ap_l"],
].copy()

f_tab["F"] = np.log(np.maximum(f_tab["xp_ap_l"], 1e-12))

vlf_tab = hrv.loc[
    hrv["record"].isin(dev["record"]) & hrv["window"].eq(180),
    ["record", "minute", "label", "corrected", "vlf"],
].copy()

datos_umbrales = f_tab.merge(
    vlf_tab,
    on=["record", "minute"],
    suffixes=("_F", "_HRV"),
    validate="one_to_one",
)

assert (
    datos_umbrales["label_F"] == datos_umbrales["label_HRV"]
).all(), "Las etiquetas de F y HRV no coinciden."

validos = (
    datos_umbrales["label_F"].isin([0, 1])
    & datos_umbrales["corrected_F"].le(0.20)
    & datos_umbrales["corrected_HRV"].le(0.20)
    & np.isfinite(datos_umbrales["F"])
    & np.isfinite(datos_umbrales["vlf"])
)

datos_umbrales = (
    datos_umbrales.loc[
        validos, ["record", "minute", "label_F", "F", "vlf"]
    ]
    .rename(columns={"label_F": "label", "vlf": "VLF"})
    .merge(dev, on="record", validate="many_to_one")
    .sort_values(["record", "minute"])
    .reset_index(drop=True)
)

y = datos_umbrales["label"].to_numpy(dtype=int)
registros = datos_umbrales["record"].to_numpy()

resultados = []
umbrales_youden = {}

for feature in ("F", "VLF"):
    scores = datos_umbrales[feature].to_numpy()
    resultado = roc_youden(scores, y)
    umbral = resultado["umbral"]
    umbrales_youden[feature] = umbral

    acuerdo = record_agreement(scores, y, registros, umbral)

    resultados.append({
        "Característica": feature,
        "Umbral Youden": umbral,
        "Sensibilidad": resultado["se"],
        "Especificidad": resultado["esp"],
        "AUC": resultado["auc"],
        "Spearman por registro": acuerdo["spearman"],
    })

print(f"Minutos comunes: {len(datos_umbrales)}")
print(f"Registros: {datos_umbrales['record'].nunique()}")
print(f"Sujetos: {datos_umbrales['subject'].nunique()}")
print(f"Normales: {(y == 0).sum()} — Apnea: {(y == 1).sum()}")

tabla_youden = pd.DataFrame(resultados).set_index("Característica")
display(tabla_youden.round(6))

Minutos comunes: 22275
Registros: 47
Sujetos: 19
Normales: 13421 — Apnea: 8854


,Umbral Youden,Sensibilidad,Especificidad,AUC,Spearman por registro
Característica,,,,,
F,-6.948641,0.842896,0.746144,0.873276,0.813371
VLF,0.001258,0.791733,0.712093,0.808133,0.416864


## Umbrales alternativos

Se calculan dos alternativas al Youden de referencia:

1. Mediana de los umbrales de Youden obtenidos en 2.000 remuestreos
   de sujetos, con semilla 2026.
2. Umbral que aproxima la proporción total de minutos clasificados
   como apnea a la proporción real en los datos de ajuste.

Los empates de puntuación se mantienen juntos: un mismo valor no
puede recibir dos clasificaciones diferentes.

Estos cálculos describen las alternativas. Su elección se realizará
mediante validación por sujetos, ajustando cada umbral exclusivamente
con los sujetos de entrenamiento de cada partición.

In [3]:
def umbral_proporcion(scores, labels):
    """Umbral cuya cantidad de positivos más se aproxima a la real.

    Ante empate se elige el menor umbral, priorizando sensibilidad.
    """
    scores = np.asarray(scores, dtype=float)
    labels = np.asarray(labels, dtype=int)

    ordenados = np.sort(scores)
    candidatos = np.r_[
        np.unique(ordenados),
        np.nextafter(ordenados[-1], np.inf),
    ]

    positivos = len(scores) - np.searchsorted(
        ordenados, candidatos, side="left"
    )
    objetivo = int(labels.sum())

    return float(candidatos[np.argmin(np.abs(positivos - objetivo))])


# Mismos sujetos remuestreados para F y VLF.
sujetos = datos_umbrales["subject"].unique()
filas_por_sujeto = [
    np.flatnonzero(datos_umbrales["subject"].to_numpy() == sujeto)
    for sujeto in sujetos
]

scores_features = {
    nombre: datos_umbrales[nombre].to_numpy()
    for nombre in ("F", "VLF")
}

rng = np.random.default_rng(2026)
bootstrap_umbrales = {"F": [], "VLF": []}

for b in range(2000):
    seleccion = rng.integers(0, len(sujetos), size=len(sujetos))
    filas = np.concatenate([filas_por_sujeto[i] for i in seleccion])
    y_boot = y[filas]

    # Youden requiere ambas clases.
    if np.unique(y_boot).size == 2:
        for nombre, scores in scores_features.items():
            bootstrap_umbrales[nombre].append(
                roc_youden(scores[filas], y_boot)["umbral"]
            )

    if (b + 1) % 500 == 0:
        print(f"{b + 1}/2000 remuestreos", flush=True)

umbrales_candidatos = {}
filas_resumen = []

for nombre, scores in scores_features.items():
    bootstrap = np.asarray(bootstrap_umbrales[nombre])
    assert len(bootstrap) > 0, "No hubo remuestreos válidos."

    candidatos = {
        "Youden": umbrales_youden[nombre],
        "Mediana bootstrap": float(np.median(bootstrap)),
        "Igualar proporción": umbral_proporcion(scores, y),
    }
    umbrales_candidatos[nombre] = candidatos

    for metodo, umbral in candidatos.items():
        pred = scores >= umbral

        sensibilidad = pred[y == 1].mean()
        especificidad = (~pred[y == 0]).mean()

        filas_resumen.append({
            "Característica": nombre,
            "Método": metodo,
            "Umbral": umbral,
            "Sensibilidad": sensibilidad,
            "Especificidad": especificidad,
            "Exactitud balanceada":
                (sensibilidad + especificidad) / 2,
            "Apnea estimada (%)": 100 * pred.mean(),
            "Spearman por registro": record_agreement(
                scores, y, registros, umbral
            )["spearman"],
        })

tabla_alternativas = pd.DataFrame(filas_resumen).set_index(
    ["Característica", "Método"]
)

print(f"\nProporción real de apnea: {100 * y.mean():.3f} %")
print(
    "Remuestreos válidos:",
    {k: len(v) for k, v in bootstrap_umbrales.items()},
)
display(tabla_alternativas.round(6))

500/2000 remuestreos
1000/2000 remuestreos
1500/2000 remuestreos
2000/2000 remuestreos

Proporción real de apnea: 39.749 %
Remuestreos válidos: {'F': 2000, 'VLF': 2000}


Umbral  Sensibilidad  Especificidad  \
Característica Método                                                      
F              Youden             -6.948641      0.842896       0.746144   
               Mediana bootstrap  -6.922476      0.837700       0.751211   
               Igualar proporción -6.528883      0.738310       0.827360   
VLF            Youden              0.001258      0.791733       0.712093   
               Mediana bootstrap   0.001259      0.791394       0.712167   
               Igualar proporción  0.001830      0.691326       0.796364   

                                   Exactitud balanceada  Apnea estimada (%)  \
Característica Método                                                         
F              Youden                          0.794520           48.799102   
               Mediana bootstrap               0.794456           48.287318   
               Igualar proporción              0.782835           39.748597   
VLF            Youden                          0.751913           48.817059   
               Mediana bootstrap               0.751781           48.799102   
               Igualar proporción              0.743845           39.748597   

                                   Spearman por registro  
Característica Método                                     
F              Youden                           0.813371  
               Mediana bootstrap                0.808571  
               Igualar proporción               0.738708  
VLF            Youden                           0.416864  
               Mediana bootstrap                0.416864  
               Igualar proporción               0.386906

## Validación de umbrales por sujetos

Se utilizan cinco grupos de sujetos, con semilla 2026. Todas las
grabaciones de una persona permanecen en el mismo grupo.

En cada partición, los tres métodos de umbral se ajustan exclusivamente
con los sujetos de entrenamiento y se aplican a los sujetos reservados.
La mediana bootstrap se calcula con 2.000 remuestreos de los sujetos
de entrenamiento.

Se evalúan también AND y OR, utilizando los umbrales de Youden
de F y VLF ajustados en entrenamiento.

Se reúnen las predicciones de los sujetos reservados para comparar
sensibilidad, especificidad, exactitud balanceada y Spearman por registro.

Esta validación compara umbrales para las características ya elegidas.
La repetición de la selección completa de características corresponde
al notebook 12.

In [4]:
import hashlib
from src.stats.compare import subject_folds

# Grupos reproducibles, conservando juntas las noches de cada sujeto.
tiene_apnea = datos_umbrales.groupby("subject")["label"].max().astype(bool)
folds = subject_folds(tiene_apnea, k=5, seed=2026)

metodos = ["Youden", "Mediana bootstrap", "Igualar proporción"]
columnas_pred = [
    f"{feature} | {metodo}"
    for feature in ("F", "VLF")
    for metodo in metodos
] + ["AND | Youden", "OR | Youden"]

# Identificador para no reutilizar resultados de otros datos o parámetros.
firma = hashlib.sha256(
    pd.util.hash_pandas_object(
        datos_umbrales, index=True
    ).to_numpy().tobytes()
    + repr(folds).encode()
    + b"umbrales_cv_v1_boot2000_seed2026"
).hexdigest()[:12]

cache_cv = ROOT / "results" / "umbrales" / firma
cache_cv.mkdir(parents=True, exist_ok=True)

predicciones_cv = pd.DataFrame(
    np.nan, index=datos_umbrales.index, columns=columnas_pred
)
umbrales_cv = []

for grupo, sujetos_fuera in enumerate(folds, 1):
    fuera = datos_umbrales["subject"].isin(sujetos_fuera).to_numpy()
    dentro = ~fuera
    indices_fuera = np.flatnonzero(fuera)
    ruta = cache_cv / f"grupo_{grupo}.npz"

    assert not (
        set(datos_umbrales.loc[dentro, "subject"])
        & set(datos_umbrales.loc[fuera, "subject"])
    )

    if ruta.exists():
        with np.load(ruta, allow_pickle=False) as guardado:
            assert np.array_equal(
                guardado["indices"], indices_fuera
            )
            pred_grupo = guardado["predicciones"]
            cortes = guardado["umbrales"]
        print(f"Grupo {grupo}/5: recuperado", flush=True)

    else:
        train = datos_umbrales.loc[dentro].reset_index(drop=True)
        y_train = train["label"].to_numpy(dtype=int)
        assert np.unique(y_train).size == 2

        sujetos_train = train["subject"].unique()
        filas_train = [
            np.flatnonzero(train["subject"].to_numpy() == s)
            for s in sujetos_train
        ]

        scores_train = {
            f: train[f].to_numpy() for f in ("F", "VLF")
        }
        boot = {"F": [], "VLF": []}
        rng = np.random.default_rng(2026 + grupo)

        for b in range(2000):
            seleccion = rng.integers(
                0, len(sujetos_train), size=len(sujetos_train)
            )
            filas = np.concatenate([filas_train[i] for i in seleccion])
            y_boot = y_train[filas]

            if np.unique(y_boot).size == 2:
                for feature in ("F", "VLF"):
                    boot[feature].append(
                        roc_youden(
                            scores_train[feature][filas], y_boot
                        )["umbral"]
                    )

            if (b + 1) % 500 == 0:
                print(
                    f"Grupo {grupo}/5 — bootstrap {b + 1}/2000",
                    flush=True,
                )

        cortes = np.empty((2, 3))
        pred_grupo = np.empty(
            (fuera.sum(), len(columnas_pred)), dtype=bool
        )

        for j, feature in enumerate(("F", "VLF")):
            assert len(boot[feature]) > 0

            cortes[j] = [
                roc_youden(scores_train[feature], y_train)["umbral"],
                float(np.median(boot[feature])),
                umbral_proporcion(scores_train[feature], y_train),
            ]

            scores_fuera = datos_umbrales.loc[
                fuera, feature
            ].to_numpy()

            for k in range(3):
                pred_grupo[:, 3 * j + k] = scores_fuera >= cortes[j, k]

        pred_grupo[:, 6] = pred_grupo[:, 0] & pred_grupo[:, 3]
        pred_grupo[:, 7] = pred_grupo[:, 0] | pred_grupo[:, 3]

        np.savez_compressed(
            ruta,
            indices=indices_fuera,
            predicciones=pred_grupo,
            umbrales=cortes,
        )

    predicciones_cv.loc[indices_fuera, columnas_pred] = (
        pred_grupo.astype(int)
    )

    for j, feature in enumerate(("F", "VLF")):
        for k, metodo in enumerate(metodos):
            umbrales_cv.append({
                "Grupo": grupo,
                "Característica": feature,
                "Método": metodo,
                "Umbral": cortes[j, k],
            })

assert predicciones_cv.notna().all().all()
predicciones_cv = predicciones_cv.astype(int)

# Cada minuto fue evaluado con un umbral ajustado sin su sujeto.
from scipy.stats import spearmanr

filas_resultados = []
for nombre in columnas_pred:
    pred = predicciones_cv[nombre].to_numpy()
    se = pred[y == 1].mean()
    esp = (1 - pred[y == 0]).mean()

    por_registro = (
        datos_umbrales[["record", "label"]]
        .assign(prediccion=pred)
        .groupby("record")[["label", "prediccion"]]
        .mean()
    )

    filas_resultados.append({
        "Método": nombre,
        "Sensibilidad": se,
        "Especificidad": esp,
        "Exactitud balanceada": (se + esp) / 2,
        "Apnea estimada (%)": 100 * pred.mean(),
        "Spearman por registro": spearmanr(
            por_registro["label"], por_registro["prediccion"]
        ).statistic,
    })

tabla_cv = pd.DataFrame(filas_resultados).set_index("Método")
display(tabla_cv.round(4))

Grupo 1/5 — bootstrap 500/2000
Grupo 1/5 — bootstrap 1000/2000
Grupo 1/5 — bootstrap 1500/2000
Grupo 1/5 — bootstrap 2000/2000
Grupo 2/5 — bootstrap 500/2000
Grupo 2/5 — bootstrap 1000/2000
Grupo 2/5 — bootstrap 1500/2000
Grupo 2/5 — bootstrap 2000/2000
Grupo 3/5 — bootstrap 500/2000
Grupo 3/5 — bootstrap 1000/2000
Grupo 3/5 — bootstrap 1500/2000
Grupo 3/5 — bootstrap 2000/2000
Grupo 4/5 — bootstrap 500/2000
Grupo 4/5 — bootstrap 1000/2000
Grupo 4/5 — bootstrap 1500/2000
Grupo 4/5 — bootstrap 2000/2000
Grupo 5/5 — bootstrap 500/2000
Grupo 5/5 — bootstrap 1000/2000
Grupo 5/5 — bootstrap 1500/2000
Grupo 5/5 — bootstrap 2000/2000


,Sensibilidad,Especificidad,Exactitud balanceada,Apnea estimada (%),Spearman por registro
Método,,,,,
F | Youden,0.8386,0.7486,0.7936,48.4804,0.8066
F | Mediana bootstrap,0.8371,0.7455,0.7913,48.6061,0.7977
F | Igualar proporción,0.7362,0.8295,0.7828,39.5331,0.7341
VLF | Youden,0.7452,0.7362,0.7407,45.5174,0.3522
VLF | Mediana bootstrap,0.7348,0.7406,0.7377,44.8395,0.3305
VLF | Igualar proporción,0.6616,0.7976,0.7296,38.4961,0.2755
AND | Youden,0.7156,0.8585,0.7871,36.9697,0.7049
OR | Youden,0.8682,0.6263,0.7472,57.0281,0.5870


In [5]:
from scipy.stats import spearmanr

comparaciones = [
    ("F | Mediana bootstrap", "F | Youden"),
    ("F | Igualar proporción", "F | Youden"),
    ("VLF | Mediana bootstrap", "VLF | Youden"),
    ("VLF | Igualar proporción", "VLF | Youden"),
    ("AND | Youden", "F | Youden"),
    ("OR | Youden", "F | Youden"),
]

nombres = list(predicciones_cv.columns)
pred = predicciones_cv.to_numpy(dtype=int)
y_cv = datos_umbrales["label"].to_numpy(dtype=int)

# Conteos por sujeto: permiten remuestrear sin copiar todos los minutos.
sujetos_ic = sorted(datos_umbrales["subject"].unique())
indice_sujeto = {s: i for i, s in enumerate(sujetos_ic)}
codigos = datos_umbrales["subject"].map(indice_sujeto).to_numpy()

n_pos = np.array([(y_cv[codigos == i] == 1).sum()
                  for i in range(len(sujetos_ic))])
n_neg = np.array([(y_cv[codigos == i] == 0).sum()
                  for i in range(len(sujetos_ic))])

tp = np.array([
    pred[(codigos == i) & (y_cv == 1)].sum(axis=0)
    for i in range(len(sujetos_ic))
])
tn = np.array([
    (1 - pred[(codigos == i) & (y_cv == 0)]).sum(axis=0)
    for i in range(len(sujetos_ic))
])

# Fracciones por registro para Spearman.
registro = datos_umbrales[["record", "subject", "label"]].copy()
registro[nombres] = pred

por_registro = registro.groupby("record").agg(
    subject=("subject", "first"),
    real=("label", "mean"),
    **{nombre: (nombre, "mean") for nombre in nombres},
)

sujeto_registro = por_registro["subject"].map(indice_sujeto).to_numpy()
real = por_registro["real"].to_numpy()
estimadas = por_registro[nombres].to_numpy()

rng = np.random.default_rng(2026)
boot_ba = []
boot_rho = []

for _ in range(2000):
    seleccion = rng.integers(
        0, len(sujetos_ic), size=len(sujetos_ic)
    )
    pesos = np.bincount(seleccion, minlength=len(sujetos_ic))

    positivos = pesos @ n_pos
    negativos = pesos @ n_neg
    if positivos == 0 or negativos == 0:
        continue

    ba = 0.5 * (
        (pesos @ tp) / positivos
        + (pesos @ tn) / negativos
    )

    filas = np.repeat(
        np.arange(len(por_registro)),
        pesos[sujeto_registro],
    )
    rho = [
        spearmanr(real[filas], estimadas[filas, j]).statistic
        for j in range(len(nombres))
    ]

    boot_ba.append(ba)
    boot_rho.append(rho)

boot_ba = np.asarray(boot_ba)
boot_rho = np.asarray(boot_rho)

resultados_ic = []

for candidata, referencia in comparaciones:
    j = nombres.index(candidata)
    k = nombres.index(referencia)

    for metrica, muestras in [
        ("Exactitud balanceada", boot_ba),
        ("Spearman por registro", boot_rho),
    ]:
        diferencias = muestras[:, j] - muestras[:, k]
        diferencias = diferencias[np.isfinite(diferencias)]
        assert len(diferencias) > 0

        lo, hi = np.percentile(diferencias, [2.5, 97.5])
        diferencia = (
            tabla_cv.loc[candidata, metrica]
            - tabla_cv.loc[referencia, metrica]
        )

        resultados_ic.append({
            "Candidata": candidata,
            "Referencia": referencia,
            "Métrica": metrica,
            "Diferencia": diferencia,
            "IC95 inferior": lo,
            "IC95 superior": hi,
        })

tabla_diferencias_cv = pd.DataFrame(resultados_ic)
display(tabla_diferencias_cv.round(4))

,Candidata,Referencia,Métrica,Diferencia,IC95 inferior,IC95 superior
0,F | Mediana bootstrap,F | Youden,Exactitud balanceada,-0.0023,-0.0072,0.0018
1,F | Mediana bootstrap,F | Youden,Spearman por registro,-0.0089,-0.0334,0.0065
2,F | Igualar proporción,F | Youden,Exactitud balanceada,-0.0108,-0.0369,0.0134
3,F | Igualar proporción,F | Youden,Spearman por registro,-0.0725,-0.1823,-0.0077
4,VLF | Mediana bootstrap,VLF | Youden,Exactitud balanceada,-0.0030,-0.0088,0.0016
5,VLF | Mediana bootstrap,VLF | Youden,Spearman por registro,-0.0216,-0.0594,0.0047
6,VLF | Igualar proporción,VLF | Youden,Exactitud balanceada,-0.0111,-0.0334,0.0105
7,VLF | Igualar proporción,VLF | Youden,Spearman por registro,-0.0767,-0.1697,0.0029
8,AND | Youden,F | Youden,Exactitud balanceada,-0.0065,-0.0385,0.0286
9,AND | Youden,F | Youden,Spearman por registro,-0.1017,-0.2635,0.0171


## Decisión

Se conserva el umbral de Youden para F y para la referencia HRV.
No se incorporan la mediana bootstrap, la igualación de proporciones
ni las combinaciones AND/OR, porque ninguna mostró una mejora
sobre su referencia en la validación por sujetos.

Con F y Youden, las predicciones de validación alcanzaron:
- Sensibilidad: 0,8386.
- Especificidad: 0,7486.
- Exactitud balanceada: 0,7936.
- Spearman por registro: 0,8066.

La proporción estimada de apnea fue del 48,48 %, frente al 39,75 %
anotado. Por lo tanto, se conserva como limitación la sobreestimación
del total de minutos con apnea en este conjunto.

Los umbrales ajustados con todo desarrollo se guardan con su precisión
completa. Se clasifica como apnea cuando la característica es mayor
o igual a su umbral.

Los intervalos presentados son exploratorios, sin ajuste por
comparaciones múltiples, y remuestrean las predicciones de validación
sin repetir el ajuste. La selección completa de características y
umbrales se evaluará en el notebook 12. El conjunto de prueba
permanece reservado.

In [6]:
import json

salida = ROOT / "results" / "umbrales"
salida.mkdir(parents=True, exist_ok=True)

configuracion = {
    "metodo": "Youden",
    "regla": "score >= umbral",
    "detector_principal": "F",
    "umbrales": {
        nombre: float(umbrales_youden[nombre])
        for nombre in ("F", "VLF")
    },
    "F": {
        "edr": "area",
        "ventana_s": 240,
        "estimador": "welch3",
        "banda_hz": [0.01, 0.05],
        "transformacion": "log(max(potencia_cruzada, 1e-12))",
    },
    "VLF": {
        "ventana_s": 180,
        "estimador": "Lomb",
        "banda_hz": [0.01, 0.04],
        "transformacion": "ninguna",
    },
    "calidad": {
        "max_fraccion_rr_corregidos": 0.20,
        "excluir_senal_perdida": False,
    },
    "qrs": "marcas de la base",
    "wiener": False,
    "minutos_ajuste": len(datos_umbrales),
    "registros_ajuste": int(datos_umbrales["record"].nunique()),
    "sujetos_ajuste": int(datos_umbrales["subject"].nunique()),
    "validacion_completa_pendiente": True,
}

ruta_config = ROOT / "docs" / "umbrales_desarrollo.json"
ruta_config.write_text(
    json.dumps(configuracion, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

tabla_youden.to_csv(salida / "youden_desarrollo.csv")
tabla_alternativas.to_csv(salida / "alternativas_desarrollo.csv")
tabla_cv.to_csv(salida / "validacion_umbrales.csv")
tabla_diferencias_cv.to_csv(
    salida / "diferencias_validacion.csv", index=False
)
pd.DataFrame(umbrales_cv).to_csv(
    salida / "umbrales_por_grupo.csv", index=False
)

print("Umbrales guardados con precisión completa:")
print(json.dumps(configuracion["umbrales"], indent=2))

Umbrales guardados con precisión completa:
{
  "F": -6.948641462292339,
  "VLF": 0.001258477270410224
}


In [12]:
import importlib
import src.stats.thresholds as thresholds

importlib.invalidate_caches()
importlib.reload(thresholds)

from src.stats.thresholds import (
    umbral_proporcion,
    mediana_youden_sujetos,
    combinar_predicciones,
)

print("Funciones importadas correctamente.")

Funciones importadas correctamente.


In [7]:
codigo_umbrales = '''

# --- Alternativas de umbral: notebook 11 ---
def _threshold_inputs(scores, labels):
    """Valida puntuaciones finitas y etiquetas binarias."""
    s = np.asarray(scores, dtype=float)
    y = np.asarray(labels)

    if s.ndim != 1 or y.ndim != 1 or len(s) != len(y) or len(s) == 0:
        raise ValueError("scores y labels deben ser vectores de igual largo.")
    if not np.isfinite(s).all() or not np.isin(y, [0, 1]).all():
        raise ValueError("Se requieren scores finitos y etiquetas 0/1.")
    return s, y.astype(int)




def mediana_youden_sujetos(scores, labels, subjects,
                          n_boot=2000, seed=2026):
    """Mediana de Youden remuestreando sujetos completos.

    Descarta remuestreos sin ambas clases. Solo deben proporcionarse
    los sujetos usados para ajustar el umbral.
    """
    s, y = _threshold_inputs(scores, labels)
    g = np.asarray(subjects)

    if g.ndim != 1 or len(g) != len(s) or pd.isna(g).any():
        raise ValueError("subjects debe identificar cada observación.")
    if np.unique(y).size != 2:
        raise ValueError("Youden requiere ambas clases.")
    if not isinstance(n_boot, (int, np.integer)) or n_boot <= 0:
        raise ValueError("n_boot debe ser un entero positivo.")

    sujetos = pd.unique(g)
    filas = [np.flatnonzero(g == sujeto) for sujeto in sujetos]
    rng = np.random.default_rng(seed)
    umbrales = []

    for _ in range(n_boot):
        seleccion = rng.integers(0, len(sujetos), size=len(sujetos))
        indices = np.concatenate([filas[i] for i in seleccion])
        if np.unique(y[indices]).size == 2:
            umbrales.append(roc_youden(s[indices], y[indices])["umbral"])

    if not umbrales:
        raise ValueError("No hubo remuestreos válidos con ambas clases.")
    return float(np.median(umbrales))


def combinar_predicciones(pred_f, pred_hrv, metodo):
    """Combina dos vectores binarios mediante AND u OR."""
    f, h = np.asarray(pred_f), np.asarray(pred_hrv)
    if f.ndim != 1 or h.shape != f.shape:
        raise ValueError("Las predicciones deben ser vectores de igual largo.")
    if not np.isin(f, [0, 1]).all() or not np.isin(h, [0, 1]).all():
        raise ValueError("Las predicciones deben ser binarias.")

    if metodo == "AND":
        return f.astype(bool) & h.astype(bool)
    if metodo == "OR":
        return f.astype(bool) | h.astype(bool)
    raise ValueError("metodo debe ser AND u OR.")
'''

ruta = ROOT / "src" / "stats" / "thresholds.py"
contenido = ruta.read_text()

funciones = [
    "_threshold_inputs",
    "umbral_proporcion",
    "mediana_youden_sujetos",
    "combinar_predicciones",
]
existen = [f"def {nombre}(" in contenido for nombre in funciones]

if not any(existen):
    ruta.write_text(contenido + codigo_umbrales)
    print("Funciones agregadas a thresholds.py.")
elif all(existen):
    print("Las funciones ya existen; no se agregaron nuevamente.")
else:
    raise RuntimeError(
        "Hay una implementación parcial. Revisar antes de agregar código."
    )

Funciones agregadas a thresholds.py.


In [9]:
codigo_tests = '''
import numpy as np
import pytest

from src.stats.thresholds import (
    roc_youden,
    umbral_proporcion,
    mediana_youden_sujetos,
    combinar_predicciones,
)


def test_proporcion_exacta():
    s = np.array([0., 1., 2., 3.])
    y = np.array([0, 0, 1, 1])
    t = umbral_proporcion(s, y)
    assert t == 2.
    assert (s >= t).sum() == y.sum()


def test_proporcion_respeta_empates():
    s = np.array([1., 1., 2., 2.])
    y = np.array([0, 0, 0, 1])
    # No se puede clasificar solo uno de los dos valores iguales a 2.
    # Dos positivos y cero positivos empatan: se prioriza sensibilidad.
    t = umbral_proporcion(s, y)
    assert t == 2.
    assert (s >= t).sum() == 2


def test_proporcion_clases_extremas():
    s = np.array([1., 2., 3.])
    assert not (s >= umbral_proporcion(s, [0, 0, 0])).any()
    assert (s >= umbral_proporcion(s, [1, 1, 1])).all()


def test_bootstrap_un_sujeto_resultado_conocido():
    s = np.array([0., 1., 2., 3.])
    y = np.array([0, 0, 1, 1])
    t = mediana_youden_sujetos(s, y, ["A"] * 4, n_boot=20)
    assert t == roc_youden(s, y)["umbral"] == 2.


def test_bootstrap_reproducible():
    s = np.array([0., 3., 1., 4., 2., 5.])
    y = np.array([0, 1, 0, 1, 0, 1])
    g = np.array(["A", "A", "B", "B", "C", "C"])
    a = mediana_youden_sujetos(s, y, g, n_boot=50, seed=2026)
    b = mediana_youden_sujetos(s, y, g, n_boot=50, seed=2026)
    assert a == b


def test_combinaciones():
    f = [0, 0, 1, 1]
    h = [0, 1, 0, 1]
    np.testing.assert_array_equal(
        combinar_predicciones(f, h, "AND"), [0, 0, 0, 1]
    )
    np.testing.assert_array_equal(
        combinar_predicciones(f, h, "OR"), [0, 1, 1, 1]
    )


def test_entradas_invalidas():
    with pytest.raises(ValueError):
        umbral_proporcion([1., np.nan], [0, 1])
    with pytest.raises(ValueError):
        mediana_youden_sujetos([1., 2.], [0, 0], ["A", "A"])
    with pytest.raises(ValueError):
        combinar_predicciones([0, 1], [0], "AND")
    with pytest.raises(ValueError):
        combinar_predicciones([0], [1], "XOR")
'''

ruta_tests = ROOT / "tests" / "test_thresholds_alternatives.py"
ruta_tests.write_text(codigo_tests)

!python -m pytest tests/test_thresholds_alternatives.py -q


ImportError: cannot import name 'umbral_proporcion' from 'src.stats.thresholds' (/content/drive/MyDrive/TP-PSB-G8-main/src/stats/thresholds.py)

In [13]:
!python -m pytest -q

.................sss.............................................x...... [ 75%]
............s...........                                                 [100%]
91 passed, 4 skipped, 1 xfailed in 71.89s (0:01:11)
